In [1]:
batch_number = 1

In [2]:
import os
import re
import json
from math import ceil
import pandas as pd
import numpy as np
import hashlib

from collections import Counter
from bson import ObjectId
from bs4 import BeautifulSoup
from fuzzywuzzy import fuzz
from datetime import datetime
from tqdm import tqdm
tqdm.pandas()

# Environment Variables
from dotenv import load_dotenv

# Llama Model
from langchain_community.llms import LlamaCpp # Llm Handler
from langchain.prompts import PromptTemplate # Prompt
from langchain_core.output_parsers import StrOutputParser # Parser

# NER Model
import spacy

# Google Maps API
import requests
import googlemaps
from geopy.distance import geodesic

# Topic Modeling
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tenacity import retry, wait_random_exponential, stop_after_attempt

# MongoDB
from pymongo import MongoClient 
from pymongo.errors import BulkWriteError

## Running Llama 3.1 8B

In [3]:
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""
    <|begin_of_text|><|start_header_id|>system<|end_header_id|>
    
    Cutting Knowledge Date: December 2023
    Today Date: 26 Jul 2024

    You are an expert in geo-location and have a deep understanding of specific places and organizations. In a short response, your task is to identify and provide the most exact real location mentioned in the news article. This can be a place, organization, facility, or any location that can help identify where the article takes place or talks about. Also, mention any specific locations or organizations explicitly found within the article that influenced your decision. If you cannot determine a location, state that explicitly. Do not discuss anthing else.<|eot_id|><|start_header_id|>user<|end_header_id|>

    You are tasked in geo-locating this news article. Generate a SHORT response specifying the most exact location you can find mentioned in the article. Give your answer in the following format:
    1. If there is one, the city the article is talking about. Otherwise, state that it can't be located. 
    2. The specific place within the city you got if you found one.
    3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision.
     
    If you cannot determine a location, state that explicitly. DO NOT MAKE UP INFORMATION. This is the news article:
    Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n <|eot_id|><|start_header_id|>assistant<|end_header_id|>""",
)


In [4]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_3_1_8B/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"

In [5]:
llm = LlamaCpp(
    model_path=llama_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
    verbose=False
)
output_parser = StrOutputParser()

In [6]:
chain = prompt | llm | output_parser

## NER Model

In [7]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

### Functions to manage caches

In [8]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

## Google Maps

In [9]:
# Load environment variables
load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [10]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [11]:
bad_locations_path = "./data_prod/bad_locations.json"
try:
    with open(bad_locations_path, 'r') as file:
        bad_locations = json.load(file)
except FileNotFoundError:
    bad_locations = []

In [12]:
ma_center_coords = (42.4072, -71.3824) 

# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
        
            coords = [latitude, longitude]

            # Get the city from the address components
            address_components = geocode_result[0]['address_components']
            city = None
            for component in address_components:
                if 'locality' in component['types']:
                    city = component['long_name']
                    break

            # Check if location couldn't be identified (it returns the center of MA)
            if geodesic(coords, ma_center_coords).km <= 2:
                bad_locations.append(location)
                return None, None, None
            elif (coords[0] == 42.3600825 and coords[1] == -71.0588801): # Check if it defaulted to the center of Boston
                bad_locations.append(location)
                return None, None, None
            else:
                return longitude, latitude, city
        else:
            print(f"[WARNING] Could not find location {location} through google maps")
            return None, None, None
    except Exception as error:
        print(f"[ERROR] Error finding locations through google maps, {error}")
        return None, None, None

## Pipeline Entry Point

### Load and Preprocess Data Set

In [13]:
# Load the data set
data_path = f"./data_set/batches/batch_{batch_number}.csv"
full_df = pd.read_csv(data_path)

In [14]:
def clean_string(str):
    return re.sub(r'[^a-zA-Z0-9]', '', str).lower()

def generate_content_id(row):
	combined_str = str(row['author']) + str(row['hl1']) + str(row['pub_date'])
	cleaned_combined_str = clean_string(combined_str)
	return hashlib.sha256(cleaned_combined_str.encode()).hexdigest()

def create_content_ids(df):
	df['content_id'] = df.apply(lambda row: generate_content_id(row), axis=1)
	df['_id'] = df['content_id']
	return

In [15]:
def partial_df(df):
    columns = df.columns

    required_columns = {'Byline': 'author',
                        'Body': 'body', 
                        "Headline": 'hl1', 
                        'Publish Date': 'pub_date', 
                        'Publisher': 'pub_name', 
                        'Paths': 'link'}
    
    partial_df = pd.DataFrame()
    for column in required_columns:
        if column in columns:
            partial_df[required_columns[column]] = df[column]
        else: 
            print(f"[ERROR] Column {column} not found in the data set")
            return None
    
    # Create unique id for each row
    create_content_ids(partial_df)
    
    # Reorder the DataFrame
    columns = ['_id'] + [col for col in partial_df.columns if col != '_id']
    partial_df = partial_df[columns]

    # Fix path link
    def fix_link(link):
        if link is None:
            return None
        if link.startswith('http'):
            return link
        if link.endswith(" (Permalink)"):
            return "https://www.wgbh.org" + link[:-11]
        else:
            return link
    
    partial_df['link'] = partial_df['link'].apply(fix_link)

    # Drop rows where at least one of the specified columns is empty
    columns_to_check = ['_id', 'hl1', 'body', 'author', 'pub_date', 'pub_name', 'link']
    partial_df = partial_df.dropna(subset=columns_to_check).reset_index(drop=True)
    
    if len(partial_df) == 0:
        return None
    
    # Drop empty rows too
    partial_df = partial_df[~partial_df['body'].apply(lambda x: isinstance(x, float))]
    partial_df = partial_df[~partial_df['hl1'].apply(lambda x: isinstance(x, float))]
    
    return partial_df

def clean_df(partial_df):
    cleaned_df = partial_df.copy()
    
    # Clean the html
    func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
    cleaned_df['body'] = partial_df['body'].apply(func_clean_html)
    cleaned_df['hl1'] = cleaned_df['hl1'].apply(func_clean_html)

    # Clean with Regex
    func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
    cleaned_df['body'] = cleaned_df['body'].apply(func_clean_regex)
    cleaned_df['hl1'] = cleaned_df['hl1'].apply(func_clean_regex)

    return cleaned_df


### Location Utility Functions

In [16]:
# Normalize a location string
def normalize_location(location):
    location = location.lower().strip() 
    
    # Remove leading terms
    leading_terms = ["a", "an", "the"]

    for term in leading_terms:
        if location.startswith(term + " "):
            location = location[len(term) + 1:]

    # Remove punctuation
    location = re.sub(r'[^\w\s]', '', location)
    
    # Expand common abbreviations
    abbreviation_map = {
        "us": "united states",
        'co': 'company',
        'pd': 'police department',
        'mit': 'massachusetts institute of technology',
        'bu': 'boston university',
        'wgbh': 'gbh',
        'cfa': 'the harvard-smithsonian center for astrophysics',
        'pbs': 'public broadcasting service',
        'ap': 'associated press',
        'aim': 'aim (alternative investment market)',
        '&': 'and',
        'npr': 'national public radio',
        'doj': 'the justice department',
        'cdc': 'the centers for disease control and prevention',
        'ssa': 'social security administration',
        'doe': 'the energy department',
        'cbpp': 'the center on budget and policy priorities',
        'mbta': 'massachusetts bay transportation authority',
        't': 'massachusetts bay transportation authority',
        'globe': 'boston globe',
        'senate': 'capitol',
        'congress': 'capitol',
        'legislature': 'capitol',
        'justice': 'department of justice',
        'house': 'u.s. house of representatives',
        'oval office': 'the white house',
        'dese': 'department of elementary and secondary education',
        'bps': 'boston public schools',
        'dhs': 'the department of homeland security',
        'fed': 'federal reserve',
        'fbi': 'the federal bureau of investigation',
        'epa': 'the environmental protection agency',
        'cdc': 'the centers for disease control and prevention',
        'nar': 'national association of realtors',
        'adl': 'anti-defamation league',
        'cbp': 'customs and border protection',
        'cia': 'central intelligence agency',
        'fda': 'food and drug administration',
        'dep': 'department of environmental protection',
        'un': 'united nations',
        'faa': 'federal aviation administration',
        'ntsb': 'national transportation safety board',
        'dua': 'department of unemployment assistance',
        'necn': 'new england cable news',
        'nar': 'national association of realtors',
        'who': 'world health organization',
        'irap': 'international refugee assistance project',
        'ncaa': 'national collegiate athletic association',
        'council': 'city council',
        'dph': 'the department of public health',
        'usda': 'the us department of agriculture',
        'bpr': 'boston public radio',
        'blm': 'black lives matter',
        'irs': 'internal revenue service',
        'necn': 'new england cable news',
        'wh': 'white house',
        'gop': 'the republican party',
        'ps': 'public schools',
        'bpd': 'boston police department',
        'ma dese': 'the massachusetts department of elementary and secondary education',
    }
    for abbr, full in abbreviation_map.items():
        location = re.sub(r'\b' + abbr + r'\b', full, location)
    
    # TODO: (maybe) Handle synonyms, variants
    # TODO: (maybe) Handle country/state/city abbreviations   
    
    # Remove extra spaces
    location = re.sub(r'\s+', ' ', location)
    
    return location

In [17]:
# Check if two locations are the same
def are_same_location(loc1, loc2, threshold=80):
    if loc1 == loc2:
        return True
    
    if loc1 in loc2 or loc2 in loc1:
        return True
    
    similarity = fuzz.token_set_ratio(loc1, loc2)
    return similarity >= threshold

# Combine two locations if they are the same
def combine_locations(location, locations):
    if (locations is None or len(locations) == 0):
        return [location]
    
    # check for news subdomains
    if "gbh" in location.split():
        location = "gbh"
    elif "npr" in location.split():
        location = "npr"

    new_locs = []

    for loc in locations:
        if are_same_location(location, loc):
            # Replace the existing location if the new one is longer
            if len(location) > len(loc):
                new_locs.append(location)
            else:
                new_locs.append(loc)
        else:
            new_locs.append(loc)

    new_locs.append(location)

    return new_locs

In [18]:
# Check if a location can be added to the list of locations
def can_add_location(location):
    unwanted_entities = load_cache("./data_prod/unwanted_locations.json")

    if location in unwanted_entities:
        return False
    elif invalid_location(location):
        return False
    else:
        return True

# Check if a location is unwanted
def invalid_location(location):
    # List of common unwanted entity types
    unwanted_places = [
        r'\bstreet\b', 
        r'\bsquare\b', 
        r'\bavenue\b', 
        r'\bboulevard\b',
        r'\broad\b', 
        r'\blane\b', 
        r'\bdrive\b', 
        r'\bdriveway\b',
        r'\bhighway\b',
        r'\bfreeway\b'
    ]
    
    # Create a combined regex pattern
    pattern = re.compile('|'.join(unwanted_places))
    
    # Check if the location matches any unwanted entity type
    if pattern.search(location):
        return True
    return False

## Get the Location Coordinates

In [19]:
known_locations_path = "./data_prod/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [20]:
# Get the coordinates of the location
def getCoordinates(location): 
    try:
        if (location == None or len(location) == 0): return None  
        # Only get coordinates if the location is not already known
        if (location in known_locations):
            longitude, latitude = known_locations[location]["coordinates"]
        else:
            # Get coordinates and save to cache
            longitude, latitude, city = callGoogleMapsAPI(location)
            if (longitude is None or latitude is None): return None
            
            known_locations[location] = {"coordinates": [longitude, latitude], "city": city, "state": None, "tract": None, "county": None}
            save_cache_to_file(known_locations, known_locations_path)

        return [longitude, latitude]
    except Exception as error:
        print(f"[ERROR] Error getting coordinates for {location}: {error}")
        return None

In [21]:
def getAllCoordinates(locations, all_locations):
    if (locations == None or len(locations) == 0): return None, all_locations

    try: 
        found_locations = {"FAC": {}, "ORG": {}}

        for type in ["FAC", "ORG"]:
            if locations[type] is None or len(locations[type]) == 0: continue
            
            unique_locations = list(set(locations[type]))
            for location in unique_locations:
                coordinates = getCoordinates(location)
                if coordinates is not None:
                    found_locations[type][location] = coordinates
        
        if len(found_locations["FAC"]) == 0 and len(found_locations["ORG"]) == 0:
            return None, locations
        else:
            return found_locations, locations
    except Exception as error:
        print(f"[ERROR] Error processing locations: {locations}, Error: {error}")
        return None


## Get Locations From Passes

In [22]:
# Get all locations from the article
def getAllLocations(article):
    locations_dict = {}
    for key in ['Explicit_Pass', 'NER_Pass', 'LLM_Pass']:
        location = article[key]
        if location is not None:
            locations_dict = location
            break
    
    if len(locations_dict) == 0:
        return None
    elif len(locations_dict["FAC"]) == 0 and len(locations_dict["ORG"]) == 0:
        return None
    else:
        return locations_dict

## Get main valid locations of article

In [23]:
def getMainLocations(article):
    all_locations = article["all_locations"]
    valid_locations = article["locations"]
    if all_locations is None or valid_locations is None: return None, None 

    full_valid_locations = {"FAC": [], "ORG": []}
    for type in ["FAC", "ORG"]:
        for location in all_locations[type]:
            if location in valid_locations[type].keys():
                full_valid_locations[type].append(location)

    main_locations = get_main_5(full_valid_locations["FAC"], full_valid_locations["ORG"])
    if main_locations is None: return None, None
    
    main_coords = []
    for location in main_locations:
        main_coords.append(valid_locations["FAC"].get(location, valid_locations["ORG"].get(location)))
    return main_locations, main_coords

# Get the top 5 most common locations
def get_main_5(facilities, organizations):
    fac_freq = Counter(facilities)
    org_freq = Counter(organizations)
    
    top_entities = [] + fac_freq.most_common(1) + org_freq.most_common(1)

    top_entities.extend([entity for entity in fac_freq.most_common() if entity not in top_entities])

    top_entities.extend([entity for entity in org_freq.most_common(5 - len(top_entities)) if entity not in top_entities])

    if len(top_entities) == 0: return None
    
    top_entities = [entity[0] for entity in top_entities]
    return top_entities

## Explicit Pass

### Locate based on Explicit Mention of Locations on Title

Load the well-known locations, organizations, and neighborhoods dictionary and the locations that we don't want to allow (i.e. Too broad or incorrect ones like "Boston", "Massachussets", etc.)

In [24]:
known_title_locs_path = "./data_prod/known_locations.json"

unwanted_entities_path = "./data_prod/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [25]:
# If a location is in the title, use that as the article's location
def get_title_entities(header):
    known_title_locs = load_cache(known_title_locs_path)
    known_title_locations = known_title_locs.keys()
    # Look through the header for known locations
    locations_list = []
    for location in known_title_locations:
        loc = normalize_location(location)
        if (loc in header and can_add_location(loc)):
            locations_list.append(loc)
        
    
    if (len(locations_list) == 0):
        return None
    else:
        all_locations = {"FAC": locations_list, "ORG": []}

        return all_locations

## NER Pass

### Identify locations of the article with NER

In [26]:
def add_entity(entity, valid_list):
    loc = normalize_location(entity)
    if (can_add_location(loc)):
        valid_list = combine_locations(loc, valid_list)
    
    return valid_list

In [27]:
# Return all valid facilities and organizations found
def get_valid_entities(entities):
    valid_facs = []
    valid_orgs = []

    if (entities is None or len(entities) == 0):
        return None

    for entity in entities:
        if (entity.label_ == "FAC"):
            valid_facs = add_entity(entity.text, valid_facs)
        elif (entity.label_ == "ORG"):
            valid_orgs = add_entity(entity.text, valid_orgs)

    if (len(valid_facs) == 0 and len(valid_orgs) == 0):
        return None
    else:
        all_locations = {"FAC": valid_facs, "ORG": valid_orgs}

        return all_locations

In [28]:
# Run NER on the body of the article and return first valid facility
def run_NER(text, truncate=True):
    if (truncate): # Truncate the text to the first 500 words
        text = ' '.join(text.split()[:500])

    if (text == None or text == ""):
        return None
    
    try:
        entities = nlp(text).ents
        all_locations = get_valid_entities(entities)
        return all_locations
        
    except Exception as error:
        print(f"[ERROR] Error running NER on text: {error}")
        return None

In [29]:
def process_NER(article):
    """
    Process the NER on the body of the article and return all valid facilities and organizations found. If 'truncate' is true, then we get the first 500 words.
    """
    if (article['Explicit_Pass'] != None): 
        return None
    else:
        all_locations = run_NER(article['body'])
        return all_locations

### Llama Prediction

In [30]:
# Run the LLM model on the title and body of the article.
def run_llm(title, body):
    try:
        llama_prediction = chain.invoke({"headline": title, "body": body})
        return llama_prediction
    except Exception as error:
        print("[ERROR] Failed running the LLM: ", error)
        return None

In [31]:
def process_LLM(article):
    """
    Try to predict the location of the article using the LLM model. Then run NER on prediction to obtain locations.
    """

    # If the article does not have an explicit location or NER location, run LLM
    if (article['Explicit_Pass'] != None):
        return None
    elif (article['NER_Pass'] != None):
        return None
    
    else:
        text = article["body"]
        text = ' '.join(text.split()[:500])
        llama_prediction = run_llm(article['hl1'], text)
        all_locations = run_NER(llama_prediction)
        return all_locations

## Geocode locations

In [32]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']
            state = results['result']['geographies']['2020 Census Blocks'][0]['STATE']
            
            return tract, county, state
        except IndexError:
            print("[ERROR] Unable to retrieve census geography for: " + location)
        except KeyError:
            print("[ERROR] Location is outside of the United States: " + location)
        except Exception as error:
            print(f"[ERROR] Error retrieving census geography for: {location}, Error: {error}")

    print("[ERROR] API call failed for: " + location + " with coordinates" + str(coordinates))
    return None, None, None  # Return this if API call failed or no tracts found

In [33]:
# Get the census tract and county of the location
def geocode(location, coordinates):
    if (location is None or len(location) == 0): return None, None, None, None  

    if (coordinates is None or len(coordinates) == 0
        or coordinates[0] is None or coordinates[1] is None): return None, None, None, None  

    # Only geocode if it's not known
    Tract = known_locations[location]["tract"]
    County = known_locations[location]["county"]
    State = known_locations[location]["state"]
    City = known_locations[location]["city"]

    if (Tract is None or County is None or State is None):
        # Geocode article
        coordinates = known_locations[location]["coordinates"]
        Tract, County, State = query_census_api(location, coordinates)

        # Save to cache
        known_locations[location]["tract"] = Tract
        known_locations[location]["county"] = County
        known_locations[location]["state"] = State
        save_cache_to_file(known_locations, known_locations_path)
    
    return Tract, County, State, City

In [34]:
def getAllGeocodes(locations, coordinates):
    tracts = []
    counties = []
    states = []
    cities = []

    if (locations == None or len(locations) == 0): return None, None, None, None
    if (coordinates == None or len(coordinates) == 0): return None, None, None, None
    
    for i, location in enumerate(locations):
        try:
            Tract, County, State, City = geocode(location, coordinates[i])
        except Exception as error:
            print(f"[ERROR] Error geocoding location: {location}, Error: {error}")
            Tract, County, State, City = None, None, None, None
            
        tracts.append(Tract)
        counties.append(County)
        states.append(State)
        cities.append(City)


    return tracts, counties, states, cities

## Get Neighborhoods

In [35]:
neigh_tract_dict = {
	"Fenway" : ["010103", "010104", "010204", "010408", "010404", "010403", "981501", "010405", "010206", "010205"],
	"Downtown": ["030302", "070202", "070102", "030301", "070104", "070103", "070201"],
	"Beacon Hill": ["020200", "020302", "020101", "981700"],
	"Dorchester" : [
	"092400", "091400", "090300", "091800", "092300", "100601", "090901", 
	"100400", "090100", "091001","090200", "100200", "091700", "092200", "090700",
	"091500", "091300", "100300", "100100", "092000", "100500", "100800", "100603",
	"091200", "100700", "092101", "091900", "091600", "091100"
	],
	"Mattapan": ["100900", "101002", "101102", "981100", "101001","101101"],
	"Jamaica Plain": [
	"120103", "981800", "110105", "120600", "120700", "120301", "081200", "120105","081101",
	"981000", "120500", "120104", "120201", "110106", "081301", "120400"
	],
	"Roslindale": ["110502", "110104", "110501", "110401", "140106", "110301", "110607", "110403","110201"],
	"Roxbury": [
	"081500", "080500", "070801", "080100", "081800", "980300", "082000", "080601", "081700", "080300",
	"090600", "081400", "090400", "070901", "082100", "081900", "081302","080401"
	],
	"West End": ["020304", "020301", "020305"],
	"Longwood": ["010300", "081001"],
	"South Boston": ["061101", "060700", "060101", "061201", "061000", "060800", "981201", "060200", "061202", "060400", "061203", "060301", "060601", "060501"],
	"Back Bay": ["010702", "010701", "010802", "010801", "010500", "010600"],
	"Charlestown": ["040100", "040300", "040401", "040600", "040801", "040200"],
	"Allston": ["000604", "000804", "000703", "000704", "000806", "000101", "000807", "000701", "000805"],
	"Hyde Park": ["140107", "140201", "140105", "980700", "140300", "140202", "140400", "140102"],
	"East Boston": ["050500", "050600", "981502", "050101", "981300", "050901", "050300", "050700", "050400", "051000", "981600", "051200", "050200", "051101"],
	"South End": ["070301", "070302", "070502", "070501", "071101", "070600", "070700", "070902", "070802", "071201", "070402"],
	"West Roxbury": ["980900", "130406", "981900", "130404", "110601", "130300", "130402", "130200", "130101"],
	"South Boston Waterfront": ["981202", "060602", "060603", "061204", "060604"],
	"North End": ["030200", "030100", "030500", "030400"],
	"Cambridge": ["354300", "354200", "353102", "353600", "352300", "354100", "359400", "353300", "353700", "353200", 
	"354601", "355000", "354602", "354000", "354901", "354902", "353900", "354700", "352102", "354500", "354800", "352600", 
	"354400", "353101", "352900", "353000", "352101", "353800", "352500", "352400", "352700", "352200", "352800", 
  	],
	"Chelsea": ["160400", "160103", "160102", "160300", "160601", "160602", "160501", "160502", "160200"],
}

In [36]:
mongo_uri = os.getenv("MONGO_URI_NAACP")
mongo_db_name = os.getenv("MONGO_DB_NAME_NAACP")
mongo_client = MongoClient(mongo_uri)


In [37]:
# Get the collection from the database
def get_collection(db_prod, collection_name):
	collection_list = db_prod.list_collection_names()

	# Initialize the collection if it doesn't exist
	if collection_name not in collection_list:
		db_prod.create_collection(collection_name)
		print(f"[INFO] Collection '{collection_name}' created.")

	return db_prod[collection_name]

# Get all neighborhoods from the database for geocoding
def get_neighborhoods(client):

    db_prod = client[mongo_db_name]

    neighborhood_collection = get_collection(db_prod, "neighborhood_data")
    tract_to_neighborhood = {}
    neighborhood_to_tracts = {}
    
    neighborhoods = neighborhood_collection.find()

    # Populate the dictionary with tract-to-neighborhood mappings
    for neighborhood in neighborhoods:
        neighborhood_name = neighborhood.get('value')
        tracts = neighborhood.get('tracts', [])
        
        for tract in tracts:
            tract_to_neighborhood[tract] = neighborhood_name

        if neighborhood_name not in neighborhood_to_tracts:
            neighborhood_to_tracts[neighborhood_name] = tracts

    return tract_to_neighborhood, neighborhood_to_tracts

In [38]:
try:
    tract_map, neigh_map = get_neighborhoods(mongo_client)
except Exception as error:
    print(f"[ERROR] Error getting neighborhoods from database: {error}")
    tract_map_path = "./data_prod/tract_map.json"
    tract_map = load_cache(tract_map_path)
    neigh_map = neigh_tract_dict
    if (tract_map == {}):
        for neigh, tracts in neigh_tract_dict.items():
            for tract in tracts:
                tract_map[tract] = neigh

        save_cache_to_file(tract_map, tract_map_path)

In [39]:
def create_neighborhood(tract, neighborhood):
    if neighborhood not in neigh_map:
        neigh_map[neighborhood] = [tract]
    else:
        neigh_map[neighborhood].append(tract)
    tract_map[tract] = neighborhood


In [40]:
def getAllNeighborhoods(articles):
    tracts = articles['tracts']

    neighborhoods = []

    if (articles['locations'] == None or len(articles['locations']) == 0): return None
    if (articles['coordinates'] == None or len(articles['coordinates']) == 0): return None
    if (tracts == None or len(tracts) == 0): return None

    for i, tract in enumerate(tracts):
        if (tract == None): 
            neighborhoods.append("No Neighborhood")
            continue

        neighborhood = tract_map.get(tract)
        if neighborhood is not None:
            neighborhoods.append(neighborhood)
        else:
            city = articles['cities'][i]
            if city is None:
                neighborhoods.append("Unknown Neighborhood")
            else:
                create_neighborhood(tract, city)
                neighborhoods.append(city)

    return neighborhoods

## Intialize tracts Demographics

In [41]:
# Get census demographics for any given article
def get_census_demographics(year, dsource, dname, tract, county, state):
    cols = 'NAME,P2_001N,P2_002N,P2_003N,P2_004N,P2_005N,P2_006N,P2_007N,P2_008N,P2_009N,P2_010N'
    base_url = f"https://api.census.gov/data/{year}/{dsource}/{dname}"

    census_url = f"{base_url}?get={cols}&for=tract:{tract}&in=county:{county}&in=state:{state}"

    census_response = requests.get(census_url)
    census_response_json = census_response.json()

    return census_response_json

def get_city_demographics(year, dsource, dname, city, state):
    cols = 'NAME,P2_001N,P2_002N,P2_003N,P2_004N,P2_005N,P2_006N,P2_007N,P2_008N,P2_009N,P2_010N'
    base_url = f"https://api.census.gov/data/{year}/{dsource}/{dname}"

    # Note: Adjust 'for' and 'in' parameters based on city-level geography
    census_url = f"{base_url}?get={cols}&for=place:*&in=state:{state}"

    census_response = requests.get(census_url)
    census_response_json = census_response.json()
    
    # Filter results to find the specific city
    city_demographics = [
        item for item in census_response_json[1:]
        if city in item[0]  # Assuming the city name is in the first column of the results
    ]
    columns = cols.split(",")
    city_demographics = [columns, city_demographics[0]]
    return city_demographics

def update_demographics(tract_collection, tract, county, state, city=None):
    try:
        if city:
            census_data = get_city_demographics("2020", "dec", "pl", city, state)
        else:
            census_data = get_census_demographics("2020", "dec", "pl", tract, county, state)
        
        if not census_data or len(census_data) < 2:
            raise ValueError("Census data is missing or malformed.")
        
        headers = census_data[0]  # Headers
        values = census_data[1]   # Data values
        data = dict(zip(headers, values))

        county_name = data.get('NAME', "")
        geoid_tract = f"{state}{county}{tract}"

        # Prepare the update document
        update_doc = {
            'demographics.p2_001n': str(data.get('P2_001N', 0)),
            'demographics.p2_002n': str(data.get('P2_002N', 0)),
            'demographics.p2_003n': str(data.get('P2_003N', 0)),
            'demographics.p2_004n': str(data.get('P2_004N', 0)),
            'demographics.p2_005n': str(data.get('P2_005N', 0)),
            'demographics.p2_006n': str(data.get('P2_006N', 0)),
            'demographics.p2_007n': str(data.get('P2_007N', 0)),
            'demographics.p2_008n': str(data.get('P2_008N', 0)),
            'demographics.p2_009n': str(data.get('P2_009N', 0)),
            'demographics.p2_010n': str(data.get('P2_010N', 0)),
            'county_name': county_name,
            'geoid_tract': geoid_tract
        }
        
        # Update MongoDB document
        tract_collection.update_one(
            {'tract': tract},
            {'$set': update_doc}
        )
        print(f"Tract {tract} updated with census data.")

    except Exception as error:
        print(f"[ERROR] Error getting census data for tract {tract}: {error}")
        tract_collection.update_one(
            {'tract': tract},
            {'$set': {"canFind": False}}
        )
        return


In [42]:
def update_tracts(tract_collection, tract, neighborhood, county, state, city, article, location):    
    # Check if the tract document exists and has demographics data
    empty_doc = {
        'tract': tract,
        'state': state,
        'county': county,
        'city': city,
        'neighborhood': neighborhood,
        'county_name': "",
        'geoid_tract': "",
        'demographics': {},
        'articles': [article],
        'locations': [location],
        'canFind': True
    }
    tract_collection.insert_one(empty_doc)
    print(f"Tract {tract} inserted into MongoDB.")
    update_demographics(tract_collection, tract, county, state)

In [43]:
def create_gen_tract(tract_collection, tract, neighborhood, county, state, city, article, location):    
    # Check if the tract document exists and has demographics data
    empty_doc = {
        'tract': tract,
        'state': state,
        'county': "",
        'city': city,
        'neighborhood': neighborhood,
        'county_name': "",
        'geoid_tract': "",
        'demographics': {},
        'articles': [article],
        'locations': [location],
        'canFind': True
    }
    tract_collection.insert_one(empty_doc)
    print(f"Tract {tract} inserted into MongoDB.")
    update_demographics(tract_collection, tract, county, state, city)

## Removing Repeated Coordinates

In [44]:
def removeRepeatedCoords(row):
    coordinates = row['coordinates']
    if coordinates is None or not isinstance(coordinates, list):
        return row
    
    coord_counts = {}
    indexes_to_remove = set()
    
    for i, coord in enumerate(coordinates):
        if coord is None or coord[0] is None or coord[1] is None:
            print(f"[WARNING] Removing unprocessed location: {row['locations'][i]}")
            indexes_to_remove.add(i)
            continue
        elif row['tracts'][i] is None:
            print(f"[WARNING] Removing location with no tract: {row['locations'][i]}")
            indexes_to_remove.add(i)
            continue
        
        coord_tuple = tuple(coord) 
        if coord_tuple in coord_counts:
            indexes_to_remove.add(i)
        else:
            coord_counts[coord_tuple] = i

    # Remove duplicates from each column
    for column in ['locations', 'coordinates', 'tracts', 'counties', 'states', 'cities', 'neighborhoods']:
        if isinstance(row[column], list):
            row[column] = [v for i, v in enumerate(row[column]) if i not in indexes_to_remove]
    

    return row


## Full Location Pipeline

In [45]:
def geolocate_articles(df):
    """
    Processes the dataaframe given by func. Does Entity Recognition and Geolocation on articles.
    
    Parameters
    ----
    df: The pandas dataframe that geolocation is being done on.

    Returns
    ---- 
    Returns a Dataframe of geolocated articles
    """
    try: 
        df["all_locations"] = None
        
        ### Explicit Mention Pass ###
        df["Explicit_Pass"] = df["hl1"].apply(get_title_entities)

        df[["Explicit_Pass", "all_locations"]] = df.apply(lambda row: pd.Series(getAllCoordinates(row["Explicit_Pass"], row["all_locations"])), axis=1)
        
        ### NER Direct Pass ### 
        df["NER_Pass"] = df.progress_apply(process_NER, axis=1) # Automatically Truncates and performs NER on first 500 words
        
        df[["NER_Pass", "all_locations"]] = df.apply(lambda row: pd.Series(getAllCoordinates(row["NER_Pass"], row["all_locations"])), axis=1)

        ### Llama + NER Inference Pass ###
        df['LLM_Pass'] = df.progress_apply(process_LLM, axis=1) # Also truncates to 500 words
        
        df[["LLM_Pass", "all_locations"]] = df.apply(lambda row: pd.Series(getAllCoordinates(row["LLM_Pass"], row["all_locations"])), axis=1)

        # Extract Locations from Passes
        df['locations'] = df.apply(getAllLocations, axis=1)

        # Get the Main Locations and separate the coordinates
        df[['locations','coordinates']] = df.apply(lambda row: pd.Series(getMainLocations(row)), axis=1)
        
        # Geocode the Coordinates (Get the Tract and County)
        df[['tracts', 'counties', 'states', 'cities']] = df.apply(lambda row: pd.Series(getAllGeocodes(row['locations'], row['coordinates'])), axis=1)

        # Get the Neighborhoods
        df["neighborhoods"] = df.apply(getAllNeighborhoods, axis=1)

        for column in ['locations', 'coordinates', 'tracts', 'counties', 'states', 'cities', 'neighborhoods']:
            df[column] = df[column].apply(lambda x: None if (x is None or len(x) == 0) else x)

        # Drop the rows that are missing information
        df = df.dropna(subset=["locations", "coordinates", "tracts", "counties", 'states', 'cities', "neighborhoods"]).reset_index(drop=True) # Clean the rows that are missing information

        # Remove repeated coordinates
        df = df.apply(removeRepeatedCoords, axis=1)

        return df
    except Exception as e: 
        print(f"[Fatal Error] geolocate_articles() ran into an Error! Data is not saved!\nRaw Error:{e}")
        raise Exception(f"FATAL ERROR {e}")
    return

## Topic Modeling

## OpenAI Client

In [46]:
openai_key = os.getenv("OPENAI_API_KEY") 
client = OpenAI(
    api_key= openai_key,
)

In [47]:
# Retry up to 10 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(10))
def get_embedding(text: str, model="text-embedding-3-small"):
    #print(text)
    try:
        embedding = client.embeddings.create(input=text, model=model).data[0].embedding
        return embedding
    except Exception as e:
        print(f"[ERROR] Failed to retrieve ADA Embedding: {e}. Replacing with replacement value!")
        return [-1.0]
    return 

## Taxonomy Lists

Content Taxanomy

In [48]:
# Get the embedding for taxonomy
taxonomy_df = pd.read_csv('./data_prod/Content_Taxonomy.csv', skiprows=5, usecols=range(8))
taxonomy_df.columns = taxonomy_df.iloc[0]
taxonomy_df = taxonomy_df.tail(-1)

tier_1_list = []
tier_2_list = []
tier_3_list = []
tier_4_list = []
for index, row in taxonomy_df.iterrows():
    if not pd.isnull(row['Tier 4']) and row['Tier 4'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_4_label = row['Tier 4']
        tier_4_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label} - {tier_4_label}')
    elif not pd.isnull(row['Tier 3']) and row['Tier 3'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_3_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label}')
    elif not pd.isnull(row['Tier 2']) and row['Tier 2'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_2_list.append(f'{tier_1_label} - {tier_2_label}')
    else:
        tier_1_label = row['Tier 1']
        tier_1_list.append(f'{tier_1_label}')

tier_1_list = list(set(tier_1_list))
tier_2_list = list(set(tier_2_list))
tier_3_list = list(set(tier_3_list))
tier_4_list = list(set(tier_4_list))

tier_1_embedding = [get_embedding(topic) for topic in tier_1_list]
tier_2_embedding = [get_embedding(topic) for topic in tier_2_list]
tier_3_embedding = [get_embedding(topic) for topic in tier_3_list]
tier_4_embedding = [get_embedding(topic) for topic in tier_4_list]

all_topics_list = []
[all_topics_list.append(topic) for topic in tier_1_list]
[all_topics_list.append(topic) for topic in tier_2_list]
[all_topics_list.append(topic) for topic in tier_3_list]
[all_topics_list.append(topic) for topic in tier_4_list]

all_topics_embedding = []
[all_topics_embedding.append(embedding) for embedding in tier_1_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_2_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_3_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_4_embedding]
print(len(all_topics_embedding))

703


Selected Taxonomy List

In [49]:
# Get embedding for the 230 topics selected by BERTopic 
selected_taxonomy_df = pd.read_csv('./data_prod/embedding_similarity_label.csv')
selected_taxonomy_df = selected_taxonomy_df.dropna(subset=['closest_topic'])
selected_topics_list = selected_taxonomy_df['closest_topic'].values.tolist()

selected_topics_embedding = [get_embedding(topic) for topic in selected_topics_list]

Client Taxonomy List

In [50]:
# Alternative taxonomy: client's list of topics
client_taxonomy_df = pd.read_excel('./data_prod/Asad_Topics_List.xlsx', names=['label'])
client_taxonomy_df['ada_embedding'] = client_taxonomy_df['label'].map(get_embedding)

In [51]:
def truncate(tokens, length=500):
    """
    Function to get the first 500 elements from a list
    """
    return tokens[:length]

### Full Topic Modeling Pipeline

In [52]:
def topic_modeling(df):
    """
    Processes dataframe and passes it to output topic labels. Does Topic Modeling task on articles.
    
    Parameters
    ----
    df: The pandas dataframe that topic modeling is being done on.

    Returns
    ---- 
    Returns a Dataframe of Topic Modeling articles
    """
    topic_df = df.copy()
    try:
        topic_df['topic_model_body'] = topic_df['body'].apply(lambda x: re.sub(re.compile('<.*?>'), '', x))
        topic_df['tokens'] = topic_df['topic_model_body'].apply(lambda x: x.split())
        topic_df['tokens'] = topic_df['tokens'].apply(truncate)
        topic_df['ada_embedding'] = topic_df.tokens.apply(lambda x: get_embedding(','.join(map(str,x)), model='text-embedding-3-small'))

        # Find most similar taxonomy (out of all toipcs) to news body
        closest_topic_list_all = []
        for index, row in topic_df.iterrows():
            target_embedding = row['ada_embedding']
            similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in all_topics_embedding]

            # Find the index of the topic with the highest similarity
            closest_topic_index = np.argmax(similarities)

            # Retrieve the closest topic embedding
            closest_topic = all_topics_list[closest_topic_index]
            closest_topic_list_all.append(closest_topic)
        topic_df['closest_topic_all'] = closest_topic_list_all

        # Find most similar taxonomy (out of 230 selected topics) to news body
        closest_topic_list_selected = []
        for index, row in topic_df.iterrows():
            target_embedding = row['ada_embedding']
            similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in selected_topics_embedding]

            # Find the index of the topic with the highest similarity
            closest_topic_index = np.argmax(similarities)

            # Retrieve the closest topic embedding
            closest_topic = selected_topics_list[closest_topic_index]
            closest_topic_list_selected.append(closest_topic)

        topic_df['closest_topic_selected'] = closest_topic_list_selected

        client_topic_embedding_list = client_taxonomy_df['ada_embedding'].to_list()
        client_topic_list = client_taxonomy_df['label'].to_list()
        similarity_arr = []

        closest_topic_list_client = []
        for index, row in topic_df.iterrows():
            target_embedding = row['ada_embedding']
            similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in client_topic_embedding_list]
            
            if max(similarities) > 0.25:    
                closest_topic_index = np.argmax(similarities) # Find the index of the topic with the highest similarity
                closest_topic = client_topic_list[closest_topic_index] # Retrieve the closest topic embedding
                closest_topic_list_client.append(closest_topic)
            else:
                closest_topic_list_client.append('Other')
            similarity_arr.append(max(similarities))
            
        topic_df['openai_labels'] = closest_topic_list_client

        return topic_df
    except Exception as e: # Loop inbounded error
        print(f"[ERROR] topic_modeling() ran into an error! \n[Raw Error]: {e}")
        raise

## Sending Articles to MongoDB

In [53]:
user_id = "org_2bHDzl2Zax0nILIzDhui2DLWdH6"
upload_id = ObjectId()

### Utility functions

In [54]:
# Convert date to a number for easier comparison
def convert_to_datesum(s):
	date_formatted = s.replace('-', '').replace(' ', '').replace(':', '')

	year = date_formatted[-4:]
	month_num = date_formatted[3:6]
	month = str(datetime.strptime(month_num, "%b").month)
	day = date_formatted[6:8]

	if (int(month) <= 9):
		year = str(year) + "0"
		return int(year + month + day)

	return int(year + month + day)

### Packing functions

In [55]:
# Pack articles and send to MongoDB
def pack_articles(db_prod, df):
	try:
		df['dateSum'] = df['pub_date'].apply(convert_to_datesum)

		article_payload = df.to_dict(orient='records')

		articles_collection = get_collection(db_prod, "articles_data")

		try: 
			articles_collection.insert_many(article_payload, ordered=False)
		except BulkWriteError as bwe:
			# Handle duplicate key errors
			write_errors = bwe.details.get('writeErrors', [])
			duplicates = [error['op'] for error in write_errors if error['code'] == 11000]
			if duplicates:
				print(f"[WARNING] Skipped {len(duplicates)} duplicate articles.")
			else:
				raise bwe
		return
	
	except Exception as err:
		raise Exception(f"[ERROR]  Error in sending Article Data\nError: {err}")
	return

# Pack the neighborhood data and send to MongoDB
def pack_neighborhoods(db_prod, df):
	try:
		neigh_collection = get_collection(db_prod, "neighborhood_data")

		# TODO: Delete this after neighborhood data is updated
		# for neighborhood in neigh_tract_dict.keys():
		# 	neigh_collection.update_one(
		# 		{'value': neighborhood},
		# 		{'$setOnInsert': {'tracts': neigh_tract_dict[neighborhood]}},
		# 		upsert = True # Creates a new document of it if it doesn't exist
		# 	)
		# print("[INFO] Neighborhoods Collection Successfully Populated!")

		# Save all new neighborhoods with associated tracts and articles
		for n, neighborhoods in enumerate(df['neighborhoods']):
			for i, neighborhood in enumerate(neighborhoods):				
				# More convoluted than it should be. There's a bug with addToSet so this is a workaround
				current_doc = neigh_collection.find_one({'value': neighborhood})
				update_data = {}
				
				if current_doc:
					if df["_id"][n] not in current_doc.get('articles', []):
						update_data['articles'] = df["_id"][n]
					if df['tracts'][n][i] not in current_doc.get('tracts', []):
						update_data['tracts'] = df['tracts'][n][i]
					if df['locations'][n][i] not in current_doc.get('locations', []):
						update_data['locations'] = df['locations'][n][i]
				else:
					# Initialize values
					update_data['articles'] = df["_id"][n]
					update_data['tracts'] = df['tracts'][n][i]
					update_data['locations'] = df['locations'][n][i]

				# Update the document if there's anything to update
				if update_data:
					neigh_collection.update_one(
						{'value': neighborhood},
						{
							'$push': update_data
						}, upsert=True
					)
	except Exception as err:
		raise Exception(f"[ERROR]  Error in sending Neighborhood Data\nError: {err}")
	return

# Pack the topics data and send to MongoDB
def pack_topics(db_prod, df):
	try:
		topic_collection = get_collection(db_prod, "topics_data")

		# Save all new topics with associated articles
		for n, topic in enumerate(df["openai_labels"]):
			topic_collection.update_one(
				{'value': topic},
				{'$addToSet': {'articles': df["_id"][n]}},
				upsert = True 
			)          
	except Exception as err:
		raise Exception(f"[ERROR]  Error in sending Topics Data\nError: {err}")
	return

# Pack the tracts data and send to MongoDB
def pack_tracts(db_prod, df):
	try:
		tract_collection = get_collection(db_prod, "tracts_data")

		# Save all new tracts with associated articles and neighborhoods
		for n, tracts in enumerate(df['tracts']):
			for i, tract in enumerate(tracts):
				if tract_collection.find_one({'tract': tract}):
					tract_collection.update_one(
					{'tract': tract},
					{
					 '$push': {'articles': df['_id'][n]},
	  				 '$addToSet': {'locations': df['locations'][n][i]},
					}, upsert=True
    				) 
				else:
					update_tracts(tract_collection, tract, df["neighborhoods"][n][i], df["counties"][n][i], df["states"][n][i], df["cities"][n][i], df['_id'][n], df['locations'][n][i]) 
				
				# Create and update global tract for neighborhood/city
				gen_tract = df["neighborhoods"][n][i]
				if tract_collection.find_one({'tract': gen_tract}):
					tract_collection.update_one(
					{'tract': gen_tract},
					{
					 '$push': {'articles': df['_id'][n]},
	  				 '$addToSet': {'locations': df['locations'][n][i]},
					}, upsert=True
					)
				else:
					create_gen_tract(tract_collection, gen_tract, gen_tract, df["counties"][n][i], df["states"][n][i], df["cities"][n][i], df['_id'][n], df['locations'][n][i])

			  
	except Exception as err:
		raise Exception(f"[ERROR] Error in sending Tracts Data\nError: {err}")
	return

# Pack the locations data and send to MongoDB
def pack_locations(db_prod, df):
	try:
		location_collection = get_collection(db_prod, "locations_data")

		# Save all new locations with associated articles
		for n, locations in enumerate(df["locations"]):
			for i, location in enumerate(locations):

				location_collection.update_one(
					{'value': location},
					{'$addToSet': {'articles': df["_id"][n]},
	  				 '$setOnInsert': {'neighborhood': df["neighborhoods"][n][i]},
					 '$setOnInsert': {'tract': df["tracts"][n][i]},
					 '$setOnInsert': {'coordinates': df["coordinates"][n][i]},
					 '$setOnInsert': {'city': df["cities"][n][i]},
	  				},
					upsert = True 
				)          
	except Exception as err:
		raise Exception(f"[ERROR]  Error in sending Locations Data\nError: {err}")
	return


### Send data to MongoDB

In [56]:
# Handle the entire process of sending data to production
def send_to_production(client, db_name, df):
	try:
		db_prod = client[db_name]

		# Pack and send all articles
		pack_articles(db_prod, df)
		pack_neighborhoods(db_prod, df)
		pack_topics(db_prod, df)
		pack_tracts(db_prod, df)
		pack_locations(db_prod, df)
		print("[INFO] Data Successfully Sent to Production!")

	except Exception as err:
		print(f"[ERROR] Error in sending data to MongoDB Prod DB\nError: {err}")
		raise Exception("Fatal Error in sending to production")
	return

## Running Everything

In [57]:
# Adjust these if would like to start/end at a different point. 
# For example if it fails at a certain point, you can start at that point again.
start_point = 0
end_point = len(full_df)
# end_point = 3
batch_size = 3

In [58]:
# Process articles in batches of 100
def process_articles(articles_df):
    results_df = pd.DataFrame()
    batch_count = (start_point // batch_size)
    batch_total = batch_count + ceil((end_point - start_point)/ batch_size)
    batch_count += 1
    total_count = 0
    for batch in range(start_point, end_point, batch_size):
        print(f"[INFO] Processing batch {batch_count} of {batch_total}")
        articles = articles_df[batch:batch + batch_size].copy()

        # Cleaning the articles
        partial = partial_df(articles)
        if partial is None or len(partial) == 0:
            print(f"[WARNING] No articles passed the cleaning pipeline")
            batch_count += 1
            continue
        
        cleaned_articles = clean_df(partial)
        print(f"[INFO] Processing {articles.shape[0]} out of {cleaned_articles.shape[0]} articles")
        
        # Conduct Entity Recognition
        print(f"[INFO] Processing through Geolocation Pipeline")
        processing_df = geolocate_articles(cleaned_articles)

        passes = ['Explicit_Pass', 'NER_Pass', 'LLM_Pass']
        for pass_ in passes:
            count = processing_df[pass_].notna().sum()
            print(f"[INFO] {count} articles passed {pass_}")
        
        if (processing_df.empty):
            print(f"[WARNING] No articles passed the geolocation pipeline")
            continue

        print(f"[INFO] Processing through Topic Modeling Pipeline")
        topic_df = topic_modeling(processing_df)

        packaged_data_df = topic_df.drop(columns=[
            'Explicit_Pass', 
            'NER_Pass', 
            'LLM_Pass', 
            'topic_model_body',
            'tokens',
            'ada_embedding',
            'closest_topic_all',
            'closest_topic_selected',
        ])

        packaged_data_df.to_csv(f"./data_results/group_{batch_number}_batch_{batch}.csv", index=False)
        total_count += packaged_data_df.shape[0]
        results_df = pd.concat([results_df, packaged_data_df])

        print("[INFO] Sending Data to MongoDB Production")
        
        packaged_data_df["userID"] = user_id
        packaged_data_df["uploadID"] = upload_id
        send_to_production(mongo_client, mongo_db_name, packaged_data_df)

        save_cache_to_file(bad_locations, bad_locations_path)
        
        batch_count += 1
        print(f"[INFO] Batch Complete! Recognized and uploaded {packaged_data_df.shape[0]} articles \n")


    print(f"[INFO] Inference Pipeline Complete! Total Articles Processed: {total_count}")
    return results_df


: 

In [59]:
results_df = process_articles(full_df)

[INFO] Processing batch 1 of 334
[INFO] Processing 3 out of 3 articles
[INFO] Processing through Geolocation Pipeline


  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [03:27<00:00, 69.08s/it]


[INFO] 0 articles passed Explicit_Pass
[INFO] 1 articles passed NER_Pass
[INFO] 0 articles passed LLM_Pass
[INFO] Processing through Topic Modeling Pipeline
[INFO] Sending Data to MongoDB Production
[INFO] Collection 'articles_data' created.
[INFO] Collection 'topics_data' created.
[INFO] Collection 'tracts_data' created.
Tract 020301 inserted into MongoDB.
Tract 020301 updated with census data.
Tract West End inserted into MongoDB.
Tract West End updated with census data.
Tract 030302 inserted into MongoDB.
Tract 030302 updated with census data.
Tract Downtown inserted into MongoDB.
Tract Downtown updated with census data.
[INFO] Collection 'locations_data' created.
[INFO] Data Successfully Sent to Production!
[INFO] Batch Complete! Recognized and uploaded 1 articles 

[INFO] Processing batch 2 of 334
[INFO] Processing 3 out of 3 articles
[INFO] Processing through Geolocation Pipeline


100%|██████████| 3/3 [01:49<00:00, 36.46s/it]


...

100%|██████████| 3/3 [01:26<00:00, 28.68s/it]


[INFO] 0 articles passed Explicit_Pass
[INFO] 2 articles passed NER_Pass
[INFO] 0 articles passed LLM_Pass
[INFO] Processing through Topic Modeling Pipeline
[INFO] Sending Data to MongoDB Production
Tract 932200 inserted into MongoDB.
Tract 932200 updated with census data.
Tract 353102 inserted into MongoDB.
Tract 353102 updated with census data.
Tract 070104 inserted into MongoDB.
Tract 070104 updated with census data.
Tract 400700 inserted into MongoDB.
Tract 400700 updated with census data.
Tract Brookline inserted into MongoDB.
Tract Brookline updated with census data.
Tract 342402 inserted into MongoDB.
Tract 342402 updated with census data.
Tract Everett inserted into MongoDB.
Tract Everett updated with census data.
Tract 020304 inserted into MongoDB.
Tract 020304 updated with census data.
[INFO] Data Successfully Sent to Production!
[INFO] Batch Complete! Recognized and uploaded 2 articles 

[INFO] Processing batch 9 of 334
[INFO] Processing 3 out of 3 articles
[INFO] Processing

100%|██████████| 3/3 [02:50<00:00, 56.92s/it]


[INFO] 0 articles passed Explicit_Pass
[INFO] 1 articles passed NER_Pass
[INFO] 0 articles passed LLM_Pass
[INFO] Processing through Topic Modeling Pipeline
[INFO] Sending Data to MongoDB Production
[INFO] Data Successfully Sent to Production!
[INFO] Batch Complete! Recognized and uploaded 1 articles 

[INFO] Processing batch 10 of 334
[INFO] Processing 3 out of 3 articles
[INFO] Processing through Geolocation Pipeline


100%|██████████| 3/3 [05:05<00:00, 107.94s/it]

In [ ]:
results_df.head(10)

In [ ]:
results_df.to_csv(f"./data_results/group_{batch_number}_results.csv", index=False)

In [ ]:
# df.to_csv(f"./results/full_benchmark_{sample_count}_samples_trial_{trial}.csv")
# time_df.to_csv(f"./results/full_benchmark_times_{sample_count}_samples_trial_{trial}.csv")